# 🤖 Phi-3 Mini + Web Search Agent
### Run cells one by one from top to bottom
> ⚠️ **First:** Go to `Runtime → Change runtime type → T4 GPU` before running anything!

In [1]:
!pip install transformers==4.41.2 accelerate duckduckgo-search -q
print('✅ Done! Now go to Runtime → Restart session, then run from Cell 2')

✅ Done! Now go to Runtime → Restart session, then run from Cell 2


In [2]:
# ✅ Cell 2 — Check GPU
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f'\n✅ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    print('\n⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU')

Sun Feb 22 15:37:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ✅ Cell 3 — All Imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from duckduckgo_search import DDGS
import textwrap

print('✅ All imports done!')
print(f'🖥️ Device: {"GPU - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow!)"}')

✅ All imports done!
🖥️ Device: GPU - Tesla T4


In [5]:
# Alternative loader using device directly
model_id = 'microsoft/Phi-3-mini-4k-instruct'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

print('⏳ Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print('⏳ Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation='eager',
).to(device)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=0 if device == 'cuda' else -1,
)

print('✅ Model loaded!')

Using device: cuda
⏳ Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


⏳ Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model loaded!


In [6]:
# ✅ Cell 5 — Web Search Function
def web_search(query: str, max_results: int = 4) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return 'No results found.'
        output = ''
        for i, r in enumerate(results, 1):
            output += f"[{i}] {r['title']}\n{r['body']}\nSource: {r['href']}\n\n"
        return output.strip()
    except Exception as e:
        return f'Search error: {e}'

print('✅ Web search function ready!')

✅ Web search function ready!


In [10]:
# ✅ Cell 6 — Smart Agent (Model decides when to search)

def generate_answer(user_query: str, context: str = '') -> str:
    if context:
        system_prompt = (
            'You are a helpful assistant. Use the web search results below '
            'to answer the user question accurately and concisely.\n\n'
            f'Search Results:\n{context}\n'
        )
    else:
        system_prompt = 'You are a helpful assistant. Answer the user question accurately.'

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_query},
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    generated = output[0]['generated_text']
    return generated[len(prompt):].strip()


def needs_search(query: str) -> bool:
    """Ask the model itself if this question needs a web search."""
    decision_prompt = f"""You are a decision-making assistant.
A user asked: "{query}"

Does this question require searching the web for up-to-date or real-time information?
Answer with only ONE word: YES or NO.

- YES if the question is about: current events, sports results, rankings, prices, news, recent happenings, who currently holds a title/position, weather, or anything that changes over time.
- NO if the question is about: general knowledge, definitions, history, math, coding, or facts that don't change.

Answer:"""

    messages = [
        {'role': 'user', 'content': decision_prompt}
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    output = pipe(
        prompt,
        max_new_tokens=5,        # only need YES or NO
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    generated = output[0]['generated_text']
    answer = generated[len(prompt):].strip().upper()
    return 'YES' in answer


def agent(query: str) -> str:
    print(f'\n🔍 Query: {query}')
    print('🤔 Deciding if web search is needed...')

    if needs_search(query):
        print('📡 Searching the web...')
        results = web_search(query)[:3000]
        print('✅ Search complete! Generating answer...')
        return generate_answer(query, context=results)
    else:
        print('🧠 Using model knowledge...')
        return generate_answer(query)

print('✅ Smart agent ready!')


✅ Smart agent ready!


In [ ]:
# ✅ Cell 7 — Start Chatting!
print('=' * 55)
print('   🤖 Phi-3 Mini + Web Search Agent')
print('   Type "quit" to exit')
print('=' * 55)

while True:
    user_input = input('\nYou: ').strip()
    if user_input.lower() in ['quit', 'exit', 'q']:
        print('👋 Goodbye!')
        break
    if not user_input:
        continue
    response = agent(user_input)
    print(f'\n🤖 Assistant:\n')
    print(textwrap.fill(response, width=80))

   🤖 Phi-3 Mini + Web Search Agent
   Type "quit" to exit

You: who has most champions league titles?


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



🔍 Query: who has most champions league titles?
🤔 Deciding if web search is needed...
🧠 Using model knowledge...

🤖 Assistant:

As of my knowledge cutoff in 2023, Real Madrid holds the record for the most
UEFA Champions League titles, with a total of 13 titles. They have won the
competition in the following years: 1956, 1957, 1958, 1959, 1960, 1966, 1998,
2000, 2002, 2014, 2016, 2017, and 2018.

You: what is photosynthesis?

🔍 Query: what is photosynthesis?
🤔 Deciding if web search is needed...
🧠 Using model knowledge...

🤖 Assistant:

Photosynthesis is a process used by plants, algae, and certain bacteria to
convert light energy, usually from the sun, into chemical energy that can be
later released to fuel the organisms' activities. This process involves the
absorption of carbon dioxide (CO2) and water (H2O) by the plant, which, in the
presence of sunlight, are converted into glucose (C6H12O6) and oxygen (O2). The
general equation for photosynthesis can be represented as:   6 CO2 + 6 